In [28]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB

g3efr = pd.read_csv('data.csv')
g3efr = g3efr.drop(['id', 'Unnamed: 32'], axis=1, errors='ignore')
g3efr['diagnosis'] = g3efr['diagnosis'].map({'M': 1, 'B': 0})
g3efr = g3efr.fillna(g3efr.mean())

In [29]:
X = g3efr.drop('diagnosis', axis=1)
y = g3efr['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



In [ ]:
from collections import defaultdict

class GaussianNB:
    """
    Mathematical Foundation:
    - Uses Bayes' Theorem: P(y|X) = P(X|y) * P(y) / P(X)
    - Assumes each feature follows a Gaussian (normal) distribution within each class
    - For each class, we calculate mean and variance for each feature
    - Prediction uses the Gaussian probability density function
    """

    def __init__(self):

        self.class_priors = {}
        self.class_means = {}
        self.class_variances = {}
        self.classes = None

    def fit(self, X, y):
        """
        Train the Gaussian Naive Bayes classifier.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
            Training data
        y : array-like, shape (n_samples,)
            Target values
        """

        X = np.array(X)
        y = np.array(y)

        self.classes = np.unique(y)
        n_features = X.shape[1]

        for c in self.classes:
            X_c = X[y == c]

            # Class prior P(y) = count(samples of class y) / total samples
            self.class_priors[c] = len(X_c) / len(X)

            # Calculate mean for each feature in this class
            self.class_means[c] = X_c.mean(axis=0)
            
            # Calculate variance for each feature in this class
            # Variance = E[(x - mean)^2]
            self.class_variances[c] = X_c.var(axis=0)

            # Add small epsilon to avoid division by zero
            self.class_variances[c] += 1e-9

    def gaussian_pdf(self, x, mean, variance):
        """
        Calculate the Gaussian (Normal) Probability Density Function.
        
        Formula:
        P(x|y) = (1 / sqrt(2π * variance)) * exp(-(x - mean)^2 / (2 * variance))
        
        Parameters:
        -----------
        x : feature value
        mean : mean of feature for a given class
        variance : variance of feature for a given class
        
        Returns:
        --------
        probability : scalar value of the Gaussian PDF
        """

        numerator = np.exp(-(x - mean) ** 2 / (2 * variance))
        denominator = np.sqrt(2 * np.pi * variance)

        return numerator / denominator
    
    def predict(self, X):
        """
        Predict class labels for samples in X.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
            Samples to predict
        
        Returns:
        --------
        predictions : array, shape (n_samples,)
            Predicted class labels
        
        Process:
        1. For each sample, calculate posterior probability for each class
        2. Return the class with highest posterior probability
        """

        X = np.array(X)
        predictions = []

        for x in X:
            posterior_probs = {}

            # For each class, calculate posterior probability P(y|X)
            for c in self.classes:
                prior = np.log(self.class_priors[c])
                posterior = prior

                # Calculate the likelihood P(X|y)
                # Using log-likelihood to avoid numerical underflow:
                # log(P(X|y)) = sum(log(P(x_i|y))) for all features
                for i, x_i in enumerate(x):
                    mean = self.class_means[c][i]
                    variance = self.class_variances[c][i]
                    posterior += np.log(self.gaussian_pdf(x_i, mean, variance))

                posterior_probs[c] = posterior

            prediction = max(posterior_probs, key=posterior_probs.get)
            predictions.append(prediction)

        return np.array(predictions)
    
    def predict_proba(self, X):
        """
        Return prediction probabilities for each class.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
            Samples to predict
        
        Returns:
        --------
        proba : array, shape (n_samples, n_classes)
            Probability estimates for each class
        """
        
        X = np.array(X)
        probabilities = []
        
        for x in X:
            log_scores = []
            
            for c in self.classes:
                score = np.log(self.class_priors[c])
                
                for i, x_i in enumerate(x):
                    mean = self.class_means[c][i]
                    variance = self.class_variances[c][i]
                    score += np.log(self.gaussian_pdf(x_i, mean, variance))
                
                log_scores.append(score)
            
            exp_scores = np.exp(log_scores)
            
            sum_scores = np.sum(exp_scores)
            prob = exp_scores / sum_scores
            
            probabilities.append(prob)
            
        return np.array(probabilities)
    
    def score(self, X, y):
        """
        Calculate accuracy score.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
            Test data
        y : array-like, shape (n_samples,)
            True labels
        
        Returns:
        --------
        accuracy : float
            Fraction of correct predictions
        """

        predictions = self.predict(X)
        accuracy = np.mean(predictions == y) * 100
        return accuracy
    
gnb = GaussianNB()
gnb.fit(X_train_scaled, y_train)

y_pred = gnb.predict(X_test_scaled)
y_proba = gnb.predict_proba(X_test_scaled)

# Calculate accuracy
accuracy = gnb.score(X_test_scaled, y_test)
print(f"Accuracy: {accuracy}")

Accuracy: 96.49122807017544
